# CELL 1: SETUP & LOAD PARQUET (VIA GOOGLE DRIVE)


In [10]:
!pip install -q gymnasium numpy pandas pyarrow

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
from google.colab import drive

print("Mounting Google Drive...")
# This connects Colab securely to your Drive
drive.mount('/content/drive')

# Pointing to the Parquet file in your Drive
PARQUET_PATH = '/content/drive/MyDrive/oxford_train_env.parquet'

print(f"\nLoading pristine Parquet dataset from: {PARQUET_PATH}")
try:
    df_train = pd.read_parquet(PARQUET_PATH)
    MIN_CAPACITY = df_train['capacity_Ah'].min()
    print(f"✓ Dataset loaded! Total rows: {len(df_train):,}")
    print("Ready to build the Dynamic Environment.")
except FileNotFoundError:
    print("❌ Error: File not found. Please ensure 'oxford_train_env.parquet' is in your main Google Drive folder.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Loading pristine Parquet dataset from: /content/drive/MyDrive/oxford_train_env.parquet
✓ Dataset loaded! Total rows: 5,423,797
Ready to build the Dynamic Environment.


# CELL 2: THE DYNAMIC GYM ENVIRONMENT

In [11]:
class DynamicBatteryEnv(gym.Env):
    def __init__(self, data):
        super(DynamicBatteryEnv, self).__init__()
        self.data = data.reset_index(drop=True)
        self.max_steps = len(self.data) - 1

        # ACTIONS: 0 (Aggressive), 1 (Moderate), 2 (Conservative)
        self.action_space = spaces.Discrete(3)

        # OBSERVATIONS: voltage_V, current_A, temperature_C, voltage_diff_V, temp_diff_C, capacity_Ah
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(6,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        # Gymnasium requires handling the random seed in reset
        super().reset(seed=seed)
        self.current_step = 0
        obs = self._get_observation()
        info = {} # Gymnasium reset requires returning an info dict
        return obs, info

    def _get_observation(self):
        obs = self.data.iloc[self.current_step][
            ['voltage_V', 'current_A', 'temperature_C', 'voltage_diff_V', 'temp_diff_C', 'capacity_Ah']
        ].values
        return np.array(obs, dtype=np.float32)

    def step(self, action):
        raw_temp_spike = self.data.iloc[self.current_step]['temp_diff_C']
        current_capacity = self.data.iloc[self.current_step]['capacity_Ah']

        # DYNAMIC PHYSICS INTERVENTION
        if action == 0:
            actual_temp_stress = raw_temp_spike
            efficiency_reward = 1.0
        elif action == 1:
            actual_temp_stress = raw_temp_spike * 0.5
            efficiency_reward = 0.5
        else: # action == 2
            actual_temp_stress = raw_temp_spike * 0.1
            efficiency_reward = 0.1

        # CALCULATE REWARD
        thermal_penalty = max(0, actual_temp_stress) * 10.0
        health_reward = current_capacity * 2.0
        reward = efficiency_reward + health_reward - thermal_penalty

        self.current_step += 1

        # Gymnasium splits 'done' into 'terminated' (natural end) and 'truncated' (forced stop)
        terminated = self.current_step >= self.max_steps
        truncated = False

        info = {
            "action_taken": action,
            "thermal_penalty": thermal_penalty,
            "reward": reward
        }

        return self._get_observation(), float(reward), terminated, truncated, info

print("✓ DynamicBatteryEnv Class created and registered using modern Gymnasium!")

✓ DynamicBatteryEnv Class created and registered using modern Gymnasium!


# CELL 3: TRADITIONAL Q-LEARNING (THE BASELINE)


In [12]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("Setting up Traditional Q-Learning Baseline...")

# 1. Discretization Function
# Traditional Q-Learning cannot handle continuous decimals.
# We multiply by 10 and round off to chunk the states into discrete "bins".
def discretize_state(obs):
    discrete_obs = np.round(obs * 10)
    return tuple(discrete_obs)

# 2. Initialize Q-Table and Hyperparameters
q_table = {}
learning_rate = 0.1
discount_factor = 0.95
epsilon = 0.1 # 10% of the time, the agent takes a random action to explore

# Initialize Environment without complex wrappers
env_baseline = DynamicBatteryEnv(df_train)
obs, info = env_baseline.reset()
discrete_state = discretize_state(obs)

print("Training Q-Learning Agent (10,000 steps)...")
q_reward_total = 0

# 3. The Core Q-Learning Loop
for step in range(10000):
    # Ensure the state exists in our Q-Table
    if discrete_state not in q_table:
        q_table[discrete_state] = np.zeros(3) # 3 possible actions

    # Epsilon-Greedy Action Selection
    if np.random.uniform(0, 1) < epsilon:
        action = env_baseline.action_space.sample() # Explore randomly
    else:
        action = np.argmax(q_table[discrete_state]) # Exploit best known action

    # Take the action in the environment
    next_obs, reward, terminated, truncated, info = env_baseline.step(action)
    next_discrete_state = discretize_state(next_obs)

    if next_discrete_state not in q_table:
        q_table[next_discrete_state] = np.zeros(3)

    # Q-Learning Math (Bellman Equation Update)
    old_value = q_table[discrete_state][action]
    next_max = np.max(q_table[next_discrete_state])

    new_value = (1 - learning_rate) * old_value + learning_rate * (reward + discount_factor * next_max)
    q_table[discrete_state][action] = new_value

    # Move time forward
    discrete_state = next_discrete_state
    q_reward_total += reward

    # If the battery dies, reset the environment
    if terminated or truncated:
        obs, info = env_baseline.reset()
        discrete_state = discretize_state(obs)

print(f"✓ Q-Learning Training Complete!")
print(f"Unique States Visited (Q-Table Size): {len(q_table)}")
print(f"Total Training Reward: {q_reward_total:.2f}")

Setting up Traditional Q-Learning Baseline...
Training Q-Learning Agent (10,000 steps)...
✓ Q-Learning Training Complete!
Unique States Visited (Q-Table Size): 2184
Total Training Reward: -3272.70


# CELL 4: DEEP Q-NETWORK (DQN) AND FINAL COMPARISON


In [13]:
!pip install -q stable-baselines3
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings("ignore", category=DeprecationWarning)

print("Initializing the Dynamic Environment for DQN...")
env_dqn = Monitor(DynamicBatteryEnv(df_train))

print("Building the Deep Q-Network (DQN)...")
model = DQN(
    "MlpPolicy",
    env_dqn,
    learning_rate=0.001,
    buffer_size=100000,
    learning_starts=1000,
    batch_size=64,
    gamma=0.99,
    verbose=0
)

print("\nStarting DQN Training (20,000 steps)...")
print("Please wait 1-2 minutes. (Progress bar disabled to prevent Colab UI crashes).")
model.learn(total_timesteps=20000, progress_bar=False)

print("✓ DQN Training Complete!")
model.save("optimized_battery_dqn")

Initializing the Dynamic Environment for DQN...
Building the Deep Q-Network (DQN)...

Starting DQN Training (20,000 steps)...
Please wait 1-2 minutes. (Progress bar disabled to prevent Colab UI crashes).
✓ DQN Training Complete!


# CELL 5: 1,000 STEP EVALUATION (FINAL SHOWDOWN)

In [14]:
print("\n" + "="*50)
print("FINAL SHOWDOWN: 1,000 STEP EVALUATION")
print("="*50)

# We use a fresh environment for testing
env_eval = DynamicBatteryEnv(df_train)

# 1. Evaluate Q-Learning (Using the q_table from Cell 3)
obs, info = env_eval.reset()
q_eval_reward = 0
discrete_state = discretize_state(obs)

for _ in range(1000):
    # Exploit the Q-table (NO random exploration here)
    if discrete_state in q_table:
        action = np.argmax(q_table[discrete_state])
    else:
        action = 0 # Default aggressive if it encounters a totally new state

    obs, reward, terminated, truncated, info = env_eval.step(action)
    discrete_state = discretize_state(obs)
    q_eval_reward += reward

# 2. Evaluate DQN (Using the Neural Network)
obs, info = env_eval.reset()
dqn_eval_reward = 0

for _ in range(1000):
    # Deterministic=True means NO random exploration. Only its best moves.
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env_eval.step(int(action))
    dqn_eval_reward += reward

print(f"Traditional Q-Learning Score: {q_eval_reward:.2f}")
print(f"Deep Q-Network (DQN) Score:   {dqn_eval_reward:.2f}")

# Calculate the exact percentage improvement
if q_eval_reward != 0:
    improvement = ((dqn_eval_reward - q_eval_reward) / abs(q_eval_reward)) * 100
    print(f"\n🚀 DQN Improvement over Q-Learning: +{improvement:.2f}%")


FINAL SHOWDOWN: 1,000 STEP EVALUATION
Traditional Q-Learning Score: 1855.35
Deep Q-Network (DQN) Score:   2553.71

🚀 DQN Improvement over Q-Learning: +37.64%


# CELL 6: DOWNLOAD THE BASELINE MODEL


In [15]:
import pickle
from google.colab import files

print("Saving Traditional Q-Learning Baseline...")
with open('q_table_baseline.pkl', 'wb') as f:
    pickle.dump(q_table, f)

print("Downloading to your local machine...")
files.download('q_table_baseline.pkl')
print("✅ Baseline saved successfully!")

Saving Traditional Q-Learning Baseline...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Baseline saved successfully!


# CELL 7: DOWNLOAD THE PROPOSED MODEL


In [16]:
print("Downloading the optimized DQN model to your local machine...")
files.download('optimized_battery_dqn.zip')
print("✅ Download initiated!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download initiated!
